# Silver Layer — IT Support Tickets

In this notebook, I use the Bronze table to clean and prepare the ticket data. I check missing values, standardize some columns and add a few fields that will be useful for the Gold layer.

## 1. Load the Bronze data

I start by reading the Bronze Delta table.

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("db_projects.data_engineering.ID001_Bronze_Tickets")
display(bronze.limit(5))

ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region,ingestion_timestamp
TCKT_000001,2024-01-31T05:14:27.000Z,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,0,android,EU,2026-10-28T11:25:54.474Z
TCKT_000002,2024-10-20T06:15:49.000Z,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,0,web,NA,2026-10-28T11:25:54.474Z
TCKT_000003,2024-06-18T21:35:54.000Z,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my changes.,Thanks for reporting this bug. We will look into this and get back to you within the next 48 hours.,null,null,0,neutral,3,0,android,MEA,2026-10-28T11:25:54.474Z
TCKT_000004,2025-12-25T15:59:52.000Z,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. Our team is reviewing this and will follow up within 24 hours.,null,null,0,positive,5,1,android,LATAM,2026-10-28T11:25:54.474Z
TCKT_000005,2023-08-27T16:08:33.000Z,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you within the next 48 hours.,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,0,web,NA,2026-10-28T11:25:54.474Z


## 2. Check the data

Before making changes, I check missing values and see whether any ticket IDs are repeated.

In [0]:
# Missing values by column
missing_values = bronze.select([
    F.sum(
        (
            F.col(c).isNull() |
            F.lower(F.trim(F.col(c).cast("string"))).isin("", "null", "nan")
        ).cast("int")
    ).alias(c)
    for c in bronze.columns
])
display(missing_values)

# Repeated ticket IDs
duplicate_ids = (
    bronze.groupBy("ticket_id")
    .count()
    .filter(F.col("count") > 1)
)
print("Repeated ticket IDs:", duplicate_ids.count())

ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region,ingestion_timestamp
0,0,0,0,0,0,0,0,0,0,0,0,39887,39887,0,0,0,0,0,0,0


Repeated ticket IDs: 0


## 3. Clean the main columns

I remove duplicate ticket IDs, make sure the creation date is a timestamp and standardize the text used for priority, status, issue type and region.

In [0]:
silver = bronze.dropDuplicates(["ticket_id"])

silver = silver.withColumn(
    "created_at", F.to_timestamp("created_at")
)

for column in ["priority", "status", "issue_type"]:
    silver = silver.withColumn(
        column, F.lower(F.trim(F.col(column)))
    )

# NA is a valid region code (North America), not a missing value
silver = silver.withColumn(
    "region", F.upper(F.trim(F.col("region")))
)

## 4. Add columns for reporting

I add a resolved flag, group the resolution times and replace region codes with readable names. I also create a CSAT column that excludes zero, since the dataset uses 0 for no survey response.

In [0]:
# Only 'resolved' means successfully resolved
silver = silver.withColumn(
    "is_resolved", F.col("status") == "resolved"
)

# A missing resolution time does not always mean a data error
silver = silver.withColumn(
    "resolution_category",
    F.when(F.col("resolution_time_hours").isNull(), "No resolution time")
     .when(F.col("resolution_time_hours") < 24, "Under 24h")
     .when(F.col("resolution_time_hours") < 48, "24-48h")
     .when(F.col("resolution_time_hours") < 72, "48-72h")
     .otherwise("Over 72h")
)

silver = silver.withColumn(
    "region_name",
    F.when(F.col("region") == "EU", "Europe")
     .when(F.col("region") == "NA", "North America")
     .when(F.col("region") == "MEA", "Middle East & Africa")
     .when(F.col("region") == "LATAM", "Latin America")
     .when(F.col("region") == "APAC", "Asia-Pacific")
     .otherwise("Unknown")
)

silver = silver.withColumn(
    "csat_valid",
    F.when(F.col("csat_score").between(1, 5), F.col("csat_score"))
     .otherwise(F.lit(None))
)

## 5. Check the results

I check whether missing resolution times match the ticket statuses, and count any regions that were not recognized.

In [0]:
display(
    silver.groupBy("status")
    .agg(
        F.count("*").alias("tickets"),
        F.sum(F.col("resolution_time_hours").isNull().cast("int"))
         .alias("missing_resolution_time")
    )
    .orderBy("status")
)

unknown_regions = silver.filter(F.col("region_name") == "Unknown")
unknown_count = unknown_regions.count()
print("Unknown regions:", unknown_count)

if unknown_count > 0:
    display(unknown_regions.groupBy("region").count())

# Basic checks before saving
print("Silver rows:", silver.count())
print("Missing ticket IDs:", silver.filter(
    F.col("ticket_id").isNull() | (F.trim(F.col("ticket_id")) == "")
).count())
print("Invalid creation dates:", silver.filter(F.col("created_at").isNull()).count())
print("Negative resolution times:", silver.filter(
    F.col("resolution_time_hours") < 0
).count())

status,tickets,missing_resolution_time
closed_no_action,9982,0
in_progress,19783,19783
on_hold,10165,10165
open,9939,9939
resolved,50131,0


Unknown regions: 0
Silver rows: 100000
Missing ticket IDs: 0
Invalid creation dates: 0
Negative resolution times: 0


## 6. Save the Silver table

I save the cleaned data as a Delta table so it can be used in the Gold notebook. The ingestion timestamp from Bronze is kept.

In [0]:
(silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") # Replace the existing table's schema with the new Silver DataFrame schema if necessary. Important to consider the impact of this on production
    .saveAsTable("db_projects.data_engineering.ID001_Silver_Tickets"))

saved_silver = spark.table("db_projects.data_engineering.ID001_Silver_Tickets")
print("Saved Silver records:", saved_silver.count())
display(saved_silver.limit(5))

Saved Silver records: 100000


ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region,ingestion_timestamp,is_resolved,resolution_category,region_name,csat_valid
TCKT_000015,2025-06-15T10:10:31.000Z,CUST_01999,education,chat,mobile_app,feature_request,low,in_progress,standard,It would be great to have mobile app support in your product.,Thanks for your feature suggestion. We will look into this and get back to you within the next 48 hours.,null,null,0,neutral,0,1,api_client,MEA,2026-10-28T11:25:54.474Z,false,No resolution time,Middle East & Africa,null
TCKT_000044,2024-06-15T23:21:03.000Z,CUST_09880,individual,in_app,login_auth,bug,high,resolved,standard,The login auth feature is not saving my changes.,Thanks for reporting this bug. We are prioritizing this and will update you as soon as possible.,Patched the bug and deployed a fix to production.,16.64,0,positive,0,0,web,LATAM,2026-10-28T11:25:54.474Z,true,Under 24h,Latin America,null
TCKT_000046,2023-10-14T13:26:23.000Z,CUST_04947,small_business,phone_transcript,analytics_dashboard,feature_request,medium,open,standard,Our team needs more advanced options for analytics dashboard.,Thanks for your feature suggestion. Our team is reviewing this and will follow up within 24 hours.,null,null,0,positive,4,0,ios,LATAM,2026-10-28T11:25:54.474Z,false,No resolution time,Latin America,4
TCKT_000049,2025-01-15T09:55:48.000Z,CUST_05542,small_business,phone_transcript,mobile_app,feature_request,urgent,on_hold,standard,Can you add an option to customize mobile app?,Thanks for your feature suggestion. This has been escalated to our on-call team for immediate attention.,null,null,0,positive,4,0,api_client,APAC,2026-10-28T11:25:54.474Z,false,No resolution time,Asia-Pacific,4
TCKT_000067,2024-10-17T01:08:49.000Z,CUST_09946,individual,web_form,billing,billing_problem,medium,open,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. Our team is reviewing this and will follow up within 24 hours.,null,null,0,neutral,0,0,desktop_app,NA,2026-10-28T11:25:54.474Z,false,No resolution time,North America,null
